# Demand Forecasting Quickstart

**Use case:** material/OEM demand planning -- intermittent, promo-driven demand.

Synthetic-only: no API keys, no external data. `forecastlens.synthetic` generates demand series with known ground truth (promo dates, disruption windows) specifically so this tutorial and the test suite can validate against a controlled series instead of eyeballing a real plot.

This notebook walks through: generating data -> a leak-safe walk-forward backtest -> `DemandForecastEvaluator` (WMAPE, bias, intermittency) -> `MASE` against a seasonal-naive baseline.

In [1]:
import numpy as np

from forecastlens.backtesting import LeakSafeWalkForwardSplitter
from forecastlens.core import ForecastResult
from forecastlens.evaluators import DemandForecastEvaluator
from forecastlens.metrics import mase
from forecastlens.synthetic import load_preset

## 1. Generate a promo-driven demand series

`promo_spike` models baseline demand with occasional sharp, promo-driven spikes -- exactly the pattern that breaks a plain seasonal-naive forecast.

In [2]:
series = load_preset('promo_spike')
print(f'{len(series.values)} periods, {len(series.metadata["promo_indices"])} promo days')
print('first promo days:', series.metadata['promo_indices'][:5])

365 periods, 14 promo days
first promo days: (5, 18, 53, 65, 80)


## 2. Leak-safe walk-forward backtest

`LeakSafeWalkForwardSplitter` guarantees `max(train_idx) < min(test_idx)` for every fold -- see `tests/unit/backtesting/test_splitter.py` for the leak-safety assertions this relies on.

In [3]:
values = series.values
timestamps = series.timestamps

splitter = LeakSafeWalkForwardSplitter(horizon=7, min_train_size=90, step=7)
splits = list(splitter.split(values))
print(f'{len(splits)} folds')

39 folds


## 3. A seasonal-naive baseline forecast

No trained model here -- the point is exercising the evaluation pipeline, not showcasing a forecasting model. Each fold's forecast simply repeats the last observed week.

In [4]:
seasonality = 7
reports, mase_scores = [], []

for split in splits:
    train_idx, test_idx = split.train_idx, split.test_idx
    y_train, y_true = values[train_idx], values[test_idx]

    reps = int(np.ceil(len(test_idx) / seasonality))
    point_forecast = np.tile(y_train[-seasonality:], reps)[: len(test_idx)]

    forecast = ForecastResult(
        timestamps=timestamps[test_idx], freq='D', point=point_forecast
    )
    reports.append(DemandForecastEvaluator().evaluate(forecast, y_true))
    mase_scores.append(mase(y_true, point_forecast, y_train, seasonality=seasonality))

print(f'Mean WMAPE across folds: {np.mean([r.wmape for r in reports]):.3f}')
print(f'Mean bias across folds:  {np.mean([r.bias for r in reports]):.3f}')
print(f'Mean MASE across folds:  {np.mean(mase_scores):.3f}')

Mean WMAPE across folds: 0.219
Mean bias across folds:  0.033
Mean MASE across folds:  0.787


A MASE below 1 means the seasonal-naive backtest forecast beat the in-sample seasonal-naive scale it's compared against; WMAPE and bias give the same story in percentage terms. None of this requires a trained model -- swap in `DartsAdapter` or `NeuralForecastAdapter` output here for a real one.

## 4. Intermittent demand: a harder case

`intermittent_spare_parts` is mostly zeros -- the pattern that breaks plain MAPE (division by zero per period). WMAPE, which divides by the *sum* of actuals, stays well-defined; `DemandForecastEvaluator` also reports the intermittency rate directly.

In [5]:
spare_parts = load_preset('intermittent_spare_parts')
values = spare_parts.values
train, test = values[:-14], values[-14:]

point_forecast = np.tile(train[-seasonality:], 2)[:14]
forecast = ForecastResult(timestamps=spare_parts.timestamps[-14:], freq='D', point=point_forecast)
report = DemandForecastEvaluator().evaluate(forecast, test)

print(f'WMAPE: {report.wmape:.2f}')
print(f'Intermittency rate: {report.intermittency_rate:.0%} of periods had zero demand')

WMAPE: 1.00
Intermittency rate: 57% of periods had zero demand


With demand this sparse, a single point forecast is the wrong lens entirely -- see `DemandForecastEvaluator(service_level_quantile=...)` for evaluating a safety-stock quantile forecast instead of a point value.